# Baebugs: source-bound protocol discovery verification

Reproducible desk-only verification of the protocol registry, 31 typed source links, contextual licenses, and the current AoK snapshot. No laboratory hardware is controlled.

In [ ]:
import json, pathlib, hashlib, sys
root=pathlib.Path.cwd()
if not (root/'protocols/catalog.json').exists() and (root.parent/'protocols/catalog.json').exists(): root=root.parent
assert (root/'protocols/catalog.json').exists(), 'Run notebook from repository root or notebooks directory'
sys.path.insert(0,str(root/'site/protocols'))
from build_catalog import build, canonical, H
pack=json.loads((root/'site/mvp/mvp.json').read_text())
resolved=build(root)
print('MVP source root:',resolved['evidence_root'])
print('Templates:',len(resolved['recipes']),'relationships:',len(resolved['relationships']))
assert len(resolved['recipes'])==9 and len(resolved['relationships'])==31

## Recompute source-file bytes and per-template SHA-256 (identity, not scientific truth)

In [ ]:
for recipe in resolved['recipes']:
    content={k:v for k,v in recipe.items() if k not in ('content_sha256','context_sha256','source_atoms','evidence_root','execution_permission','signed')}
    assert H(canonical(content))==recipe['content_sha256']
    for atom in recipe['source_atoms']:
        actual=hashlib.sha256((root/atom['path']).read_bytes()).hexdigest()
        assert actual==atom['sha256']
print('PASS: template identities and all 31 source byte digests')

## Verify typed derived edges and persisted failure boundaries

In [ ]:
for relationship in resolved['relationships']:
    edge={k:v for k,v in relationship.items() if k!='edge_sha256'}
    assert H(canonical(edge))==relationship['edge_sha256']
assert pack['custody']['breaks']==25
assert pack['custody']['result_breaks']==8
assert pack['custody']['withheld_photo_breaks']==16
assert len(pack['custody']['withheld_bench_sources'])==1
assert all(x['execution_permission']=='GUIDED_BROWSER_REVIEW_ONLY' for x in resolved['recipes'])
print('PASS: 31 typed edges; 25 preexisting gaps preserved; physical NOT_TESTED')

## Admission decision

Source identity verification PASSES, but physical workflows remain NOT_TESTED. This notebook is an implementation evidence check, not an MMR commitment, signed approval, biological finding or equipment qualification.